# Usando los datos:

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:
datos = pd.DataFrame({
    'Estudiante': range(1, 11),
    'Horas': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Nota': [3.2, 4.5, 5.0, 6.8, 7.0, 8.5, 9.0, 9.2, 9.8, 10.0]
})

In [ ]:
# definiendo variables
x = datos['Horas'].to_numpy(dtype=float)
y = datos['Nota'].to_numpy(dtype=float)

n = len(x)
p = 1
alpha = 0.05

1. Usando MCO o la fórmula cerrada, encuentra tu modelo lineal.

In [ ]:
x_bar = x.mean()
y_bar = y.mean()

Sxx = np.sum((x - x_bar) ** 2)
Sxy = np.sum((x - x_bar) * (y - y_bar))

print(f"x_barra = {x_bar}")
print(f"y_barra = {y_bar}")
print(f"Sxx     = {Sxx}")
print(f"Sxy     = {Sxy}")

x_barra = 5.5
y_barra = 7.3
Sxx     = 82.5
Sxy     = 63.699999999999996


In [ ]:
b_1 = Sxy / Sxx
b_0 = y_bar - b_1 * x_bar

y_hat = b_0 + b_1 * x
residuos = y - y_hat

print(f"beta0_hat = {b_0}")
print(f"beta1_hat = {b_1}")
print(f"y_hat = {b_0} + {b_1} * x")

beta0_hat = 3.0533333333333337
beta1_hat = 0.7721212121212121
y_hat = 3.0533333333333337 + 0.7721212121212121 * x


In [ ]:
X = np.column_stack([np.ones(n), x])
b_matriz = np.linalg.inv(X.T @ X) @ X.T @ y

print("Fórmula cerrada :", np.array([b_0, b_1]))
print("Vía matricial   :", b_matriz)
print("¿Coinciden?     :", np.allclose([b_0, b_1], b_matriz))

Fórmula cerrada : [3.05333333 0.77212121]
Vía matricial   : [3.05333333 0.77212121]
¿Coinciden?     : True


In [ ]:
tabla_ajuste = datos.copy()
tabla_ajuste['y_hat']    = y_hat
tabla_ajuste['residuo']  = residuos
tabla_ajuste['residuo2'] = residuos ** 2

tabla_ajuste

,Estudiante,Horas,Nota,y_hat,residuo,residuo2
0,1,1,3.2,3.825455,-0.625455,0.391193
1,2,2,4.5,4.597576,-0.097576,0.009521
2,3,3,5.0,5.369697,-0.369697,0.136676
3,4,4,6.8,6.141818,0.658182,0.433203
4,5,5,7.0,6.913939,0.086061,0.007406
5,6,6,8.5,7.686061,0.813939,0.662497
6,7,7,9.0,8.458182,0.541818,0.293567
7,8,8,9.2,9.230303,-0.030303,0.000918
8,9,9,9.8,10.002424,-0.202424,0.040976
9,10,10,10.0,10.774545,-0.774545,0.599921


2. Crea tu tabla ANOVA

In [ ]:
SST = np.sum((y - y_bar) ** 2)
SSR = np.sum((y_hat - y_bar) ** 2)
SSE = np.sum((y - y_hat) ** 2)

print(f"SST = {SST}")
print(f"SSR = {SSR}")
print(f"SSE = {SSE}")

print(f"\nVerificación: {SST:.4f} = {SSR:.4f} + {SSE:.4f} = {SSR + SSE:.4f}")
print("¿SST = SSR + SSE?:", np.isclose(SST, SSR + SSE))

SST = 51.760000000000005
SSR = 49.184121212121205
SSE = 2.575878787878786

Verificación: 51.7600 = 49.1841 + 2.5759 = 51.7600
¿SST = SSR + SSE?: True


In [ ]:
df_reg = p
df_error = n - p - 1
df_total = n - 1

print(f"Grados de libertad regresión = p = {df_reg}")
print(f"Grados de libertad error = n - p - 1 = {df_error}")
print(f"Grados de libertad total = n - 1     = {df_total}")
print(f"Total = error + regresión {df_total} = {df_reg} + {df_error} =", df_total == df_reg + df_error)

Grados de libertad regresión = p = 1
Grados de libertad error = n - p - 1 = 8
Grados de libertad total = n - 1     = 9
Total = error + regresión 9 = 1 + 8 = True


In [ ]:
MSR = SSR / df_reg
MSE = SSE / df_error
F_stat = MSR / MSE

print(f"MSR = SSR/p = {MSR}")
print(f"MSE = SSE/(n-p-1) = {MSE} = estimación de sigma^2")
print(f"sigma_hat = sqrt(MSE) = {np.sqrt(MSE)}")

MSR = SSR/p = 49.184121212121205
MSE = SSE/(n-p-1) = 0.32198484848484826 = estimación de sigma^2
sigma_hat = sqrt(MSE) = 0.5674370876888893


In [ ]:
anova = pd.DataFrame({
    'Fuente': ['Regresión', 'Error', 'Total'],
    'SS':     [SSR, SSE, SST],
    'df':     [df_reg, df_error, df_total],
    'MS':     [MSR, MSE, np.nan],
    'F':      [F_stat, np.nan, np.nan]
}).set_index('Fuente')

anova

,SS,df,MS,F
Fuente,,,,
Regresión,49.184121,1,49.184121,152.752906
Error,2.575879,8,0.321985,NaN
Total,51.760000,9,NaN,NaN


In [ ]:
R2 = SSR / SST
r_pearson = Sxy / np.sqrt(Sxx * np.sum((y - y_bar) ** 2))

print(f"R^2 = {R2} = % de variabilidad")
print(f"r de Pearson = {r_pearson}")
print(f"r^2 = {r_pearson**2} = ¿igual a R^2?", np.isclose(R2, r_pearson**2))

R^2 = 0.9502341810688022 = % de variabilidad
r de Pearson = 0.97479955943199
r^2 = 0.9502341810688019 = ¿igual a R^2? True


3. Realiza Prueba F y t

In [ ]:
F_critico = stats.f.ppf(1 - alpha, df_reg, df_error)
p_valor_F = stats.f.sf(F_stat, df_reg, df_error)

print(f"F observado = {F_stat:.4f}")
print(f"F crítico (0.05, 1, 8)  = {F_critico:.4f}")
print(f"p-valor = {p_valor_F:.3e}")

F observado = 152.7529
F crítico (0.05, 1, 8)  = 5.3177
p-valor = 1.712e-06


In [ ]:
rechazo_por_critico = F_stat > F_critico
rechazo_por_pvalor  = p_valor_F < alpha

print(f"¿F > F_crítico = {F_stat:.4f} > {F_critico:.4f} = {rechazo_por_critico}")
print(f"¿p-valor < alpha = {p_valor_F:.3e} < {alpha} = {rechazo_por_pvalor}")

¿F > F_crítico = 152.7529 > 5.3177 = True
¿p-valor < alpha = 1.712e-06 < 0.05 = True


In [ ]:
print("Rechazamos H_0: el modelo es globalmente significativo.")

Rechazamos H_0: el modelo es globalmente significativo.


In [ ]:
SE_beta1 = np.sqrt(MSE / Sxx)
SE_beta0 = np.sqrt(MSE * (1/n + x_bar**2 / Sxx))

print(f"SE(beta1_hat) = sqrt(MSE/Sxx) = {SE_beta1}")
print(f"SE(beta0_hat) = sqrt(MSE(1/n + x_bar²/Sxx)) = {SE_beta0}")

SE(beta1_hat) = sqrt(MSE/Sxx) = 0.062472767253429644
SE(beta0_hat) = sqrt(MSE(1/n + x_bar²/Sxx)) = 0.38763332668850325


In [ ]:
gl = n - 2

t_b1 = b_1 / SE_beta1
t_b0 = b_0 / SE_beta0

t_critico = stats.t.ppf(1 - alpha/2, gl)
p_valor_t1 = 2 * stats.t.sf(abs(t_b1), gl)
p_valor_t0 = 2 * stats.t.sf(abs(t_b0), gl)

t_coef = pd.DataFrame({
    'Coeficiente': ['beta 0 (intercepto)', 'beta 1 (pendiente)'],
    'Estimación':  [b_0, b_1],
    'Error Est.':  [SE_beta0, SE_beta1],
    't':           [t_b0, t_b1],
    'p-valor':     [p_valor_t0, p_valor_t1],
    'Significativo': [abs(t_b0) > t_critico, abs(t_b1) > t_critico]
})

t_coef

,Coeficiente,Estimación,Error Est.,t,p-valor,Significativo
0,beta 0 (intercepto),3.053333,0.387633,7.876860,0.000049,True
1,beta 1 (pendiente),0.772121,0.062473,12.359325,0.000002,True


4. Verifica la relación entre F y t

In [ ]:
t1_cuadrado = t_b1 ** 2
diferencia = abs(F_stat - t1_cuadrado)

print(f"F = {F_stat}")
print(f"t_beta1^2 = ({t_b1})^2 = {t1_cuadrado}")
print(f"Diferencia absoluta = {diferencia:.2e}")
print("¿F == t^2?:", np.isclose(F_stat, t1_cuadrado))

F = 152.75290574561205
t_beta1^2 = (12.35932464763395)^2 = 152.75290574561205
Diferencia absoluta = 0.00e+00
¿F == t^2?: True


5. Encuentra los intervalos de confianza.


In [ ]:
me_beta0 = t_critico * SE_beta0
me_beta1 = t_critico * SE_beta1

IC_beta0 = (b_0 - me_beta0, b_0 + me_beta0)
IC_beta1 = (b_1 - me_beta1, b_1 + me_beta1)

print(f"IC 95% beta0: {b_0:.4f} ± {t_critico:.3f}*{SE_beta0:.4f} = "
      f"[{IC_beta0[0]:.4f}, {IC_beta0[1]:.4f}]")
print(f"IC 95% beta1: {b_1:.4f} ± {t_critico:.3f}*{SE_beta1:.4f} = "
      f"[{IC_beta1[0]:.4f}, {IC_beta1[1]:.4f}]")

IC 95% beta0: 3.0533 ± 2.306*0.3876 = [2.1594, 3.9472]
IC 95% beta1: 0.7721 ± 2.306*0.0625 = [0.6281, 0.9162]


In [ ]:
contiene_cero_b0 = IC_beta0[0] <= 0 <= IC_beta0[1]
contiene_cero_b1 = IC_beta1[0] <= 0 <= IC_beta1[1]

tabla_IC = pd.DataFrame({
    'Coeficiente':   ['beta0', 'beta1'],
    'Estimación':    [b_0, b_1],
    'Límite Inf':    [IC_beta0[0], IC_beta1[0]],
    'Límite Sup':    [IC_beta0[1], IC_beta1[1]],
    '¿Contiene 0?':  [contiene_cero_b0, contiene_cero_b1],
    'Significativo': [not contiene_cero_b0, not contiene_cero_b1]
}).set_index('Coeficiente')

tabla_IC

,Estimación,Límite Inf,Límite Sup,¿Contiene 0?,Significativo
Coeficiente,,,,,
beta0,3.053333,2.159449,3.947217,False,True
beta1,0.772121,0.628059,0.916184,False,True


In [ ]:
print("El intervalo NO contiene 0 -> rechazamos H0: beta1 = 0.")

El intervalo NO contiene 0 -> rechazamos H0: beta1 = 0.


6. Crea la tabla de IC de la media y IP

In [ ]:
SE_median = np.sqrt(MSE * (1/n + ((x - x_bar)**2) / Sxx))
SE_pred = np.sqrt(MSE * (1 + 1/n + ((x - x_bar)**2) / Sxx))

ic_media_inf = y_hat - t_critico * SE_median
ic_media_sup = y_hat + t_critico * SE_median

ip_inf = y_hat - t_critico * SE_pred
ip_sup = y_hat + t_critico * SE_pred

tabla_ic_ip = pd.DataFrame({
    'Estudiante': datos['Estudiante'],
    'Horas (x)': x,
    'Nota Real (y)': y,
    'y_hat': y_hat,
    'IC_Media_Inf': ic_media_inf,
    'IC_Media_Sup': ic_media_sup,
    'IP_Inf': ip_inf,
    'IP_Sup': ip_sup
})

tabla_ic_ip

,Estudiante,Horas (x),Nota Real (y),y_hat,IC_Media_Inf,IC_Media_Sup,IP_Inf,IP_Sup
0,1,1.0,3.2,3.825455,3.056371,4.594538,2.307662,5.343247
1,2,2.0,4.5,4.597576,3.945305,5.249846,3.135501,6.059650
2,3,3.0,5.0,5.369697,4.821123,5.918270,3.950846,6.788548
3,4,4.0,6.8,6.141818,5.675002,6.608634,4.752530,7.531106
4,5,5.0,7.0,6.913939,6.493929,7.333950,5.539671,8.288208
5,6,6.0,8.5,7.686061,7.266050,8.106071,6.311792,9.060329
6,7,7.0,9.0,8.458182,7.991366,8.924998,7.068894,9.847470
7,8,8.0,9.2,9.230303,8.681730,9.778877,7.811452,10.649154
8,9,9.0,9.8,10.002424,9.350154,10.654695,8.540350,11.464499
9,10,10.0,10.0,10.774545,10.005462,11.543629,9.256753,12.292338


In [ ]:
from google.colab import files
from nbconvert import HTMLExporter
import nbformat

# Cargar el archivo .ipynb en el cuaderno
direccion_nombre ='/content/drive/MyDrive/Actividad_2.ipynb' # Reemplaza con el nombre de tu cuaderno

# Leer el archivo .ipynb

with open(direccion_nombre) as f:
     contenido = f.read()

# Convertir a HTML
leer_cuaderno = nbformat.reads(contenido, as_version=4)
html_exportar = HTMLExporter()

#html_exporter.exclude_input = False  # Excluye el código si es necesario
html_contenido, _ = html_exportar.from_notebook_node(leer_cuaderno)

# Guardar el archivo HTML

with open('/content/drive/MyDrive/Actividad_2.html', 'w') as f:
    f.write(html_contenido)

# Descargar el archivo HTML
files.download('/content/drive/MyDrive/Actividad_2.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>